# ⚡ Duino API — Google Colab Notebook

**Cloud-agnostic hyperscale AI inference platform — Gemma 4**

> **Runtime:** Go to `Runtime → Change runtime type → GPU (T4)` for best performance.

---

## What this notebook does
1. Clones the Duino API repository
2. Runs the cross-platform setup script (auto-detects GPU, installs deps)
3. Launches the inference API + React UI
4. Embeds the chat UI as an interactive iframe inside this notebook
5. Shows how to run a **full React project** inside Colab

---
## 📦 Cell 1 — Clone & Setup
_Run once. Takes 2–5 minutes depending on GPU availability._

In [ ]:
# ── Clone repository ──────────────────────────────────────────────────────────
import os

REPO_URL  = "https://github.com/jalalmansour/Duino_API.git"
REPO_DIR  = "/content/Duino_API"

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    print("✅ Repo already cloned — pulling latest...")
    !git -C {REPO_DIR} pull

%cd {REPO_DIR}
print(f"Working directory: {os.getcwd()}")

In [ ]:
# ── Run universal setup script ────────────────────────────────────────────────
# This handles: Python deps, GPU/CPU PyTorch, Node.js, .env, package install
!chmod +x studio/setup.sh && ./studio/setup.sh

---
## 🔑 Cell 2 — (Optional) Set Secrets
_Paste your tokens here OR use Colab Secrets (🔑 icon in the sidebar)._

In [ ]:
import os

# ── Option A: Colab Secrets (recommended) ─────────────────────────────────────
# Add HF_TOKEN and NGROK_AUTHTOKEN in the 🔑 Secrets panel → they load automatically.

# ── Option B: Paste directly (will show in output — use Secrets instead) ──────
# os.environ["HF_TOKEN"]        = "hf_xxx"   # HuggingFace token (for gated models)
# os.environ["NGROK_AUTHTOKEN"] = "xxx"       # ngrok token (free at ngrok.com)

# ── Verify ────────────────────────────────────────────────────────────────────
hf_set    = bool(os.environ.get("HF_TOKEN"))
ngrok_set = bool(os.environ.get("NGROK_AUTHTOKEN"))
print(f"HF_TOKEN set     : {'✅' if hf_set    else '⚠️  Not set (Gemma 4 may require login)'}")
print(f"NGROK_AUTHTOKEN  : {'✅' if ngrok_set else '⚠️  Not set (will use Colab proxy instead)'}")

---
## 🚀 Cell 3 — Start Platform
_Launches the API, React UI, and opens a public HTTPS URL._

In [ ]:
import sys, time
sys.path.insert(0, "/content/Duino_API")

from studio.backend.colab import start

# Start everything — returns URLs and embed HTML
result = start(
    model="gemma-4-2b",   # change to "gemma-4-9b" or "gemma-4-27b" if you have more VRAM
    api_port=8000,
    ui_port=3000,
    expose=True,           # create public HTTPS URL
)

api_url   = result["api_url"]
ui_url    = result["ui_url"]
api_key   = result["api_key"]
embed_html = result["embed_html"]

print(f"\n📡 API  : {api_url}")
print(f"🎨 UI   : {ui_url}")
print(f"🔑 Key  : {api_key}")
print(f"📖 Docs : {api_url}/docs")

---
## 🎨 Cell 4 — Embed React UI as iframe
_The full chat interface, rendered inside this notebook cell._

In [ ]:
from google.colab import output as colab_output
from IPython.display import HTML, display

# Method A: serve_kernel_port (Colab native — most reliable)
colab_output.serve_kernel_port_as_iframe(3000, height=700, width="100%")

In [ ]:
# Method B: direct iframe with postMessage configuration
from IPython.display import HTML, display

display(HTML(embed_html))

---
## 🧪 Cell 5 — Test API via Python
_Verify the inference endpoint works programmatically._

In [ ]:
import requests

# ── Single request ────────────────────────────────────────────────────────────
resp = requests.post(
    f"{api_url}/v1/chat/completions",
    headers={"X-API-Key": api_key},
    json={
        "model": "gemma-4-2b",
        "messages": [{"role": "user", "content": "What is the capital of France? Answer in one sentence."}],
        "max_tokens": 64,
        "temperature": 0.3,
    },
)

print("Status:", resp.status_code)
print("Answer:", resp.json()["choices"][0]["message"]["content"])
print("Tokens:", resp.json()["usage"])

In [ ]:
# ── Streaming request ─────────────────────────────────────────────────────────
import requests, sys

with requests.post(
    f"{api_url}/v1/chat/completions",
    headers={"X-API-Key": api_key},
    json={
        "model": "gemma-4-2b",
        "messages": [{"role": "user", "content": "Write a short poem about AI."}],
        "max_tokens": 128,
        "stream": True,
    },
    stream=True,
) as r:
    for line in r.iter_lines():
        if line:
            line = line.decode()
            if line.startswith("data: ") and line[6:] != "[DONE]":
                import json
                delta = json.loads(line[6:]).get("choices",[{}])[0].get("delta",{}).get("content","")
                print(delta, end="", flush=True)
print()

---
## 💬 Cell 6 — Session-based Multi-turn Chat
_Persistent conversation with memory._

In [ ]:
import requests

# Create a session
sess = requests.post(
    f"{api_url}/v1/sessions",
    headers={"X-API-Key": api_key},
    json={"model_id": "gemma-4-2b"},
).json()
session_id = sess["session_id"]
print(f"Session: {session_id}")

# Multi-turn conversation
def chat(message: str) -> str:
    r = requests.post(
        f"{api_url}/v1/chat/completions",
        headers={"X-API-Key": api_key},
        json={
            "model": "gemma-4-2b",
            "messages": [{"role": "user", "content": message}],
            "session_id": session_id,
            "max_tokens": 256,
        },
    ).json()
    return r["choices"][0]["message"]["content"]

print("User: My name is Jalal.")
print("AI  :", chat("My name is Jalal."))
print()
print("User: What is my name?")
print("AI  :", chat("What is my name?"))  # Should remember "Jalal"

---
## ⚛️ Cell 7 — Run a Full React Project Inside Colab
_Scaffold a brand new Vite + React app and open it as an iframe._

In [ ]:
import subprocess, threading, time, os
from IPython.display import display, HTML
from google.colab import output as colab_output

REACT_PORT = 4000
REACT_DIR  = "/content/my-react-app"

# ── Scaffold new Vite React app (skip if already exists) ──────────────────────
if not os.path.exists(REACT_DIR):
    print("Scaffolding Vite + React app...")
    !npm create vite@latest my-react-app -- --template react 2>/dev/null
    os.chdir(REACT_DIR)
    !npm install --silent
    print("✅ React app created")
else:
    os.chdir(REACT_DIR)
    print("✅ App already exists")

# ── Start Vite dev server in background ───────────────────────────────────────
def _vite():
    subprocess.run(
        ["npm", "run", "dev", "--", "--host", "0.0.0.0", "--port", str(REACT_PORT)],
        cwd=REACT_DIR, check=False,
    )

threading.Thread(target=_vite, daemon=True).start()
time.sleep(6)
print(f"Vite dev server running on port {REACT_PORT}")

# ── Expose as iframe ──────────────────────────────────────────────────────────
colab_output.serve_kernel_port_as_iframe(REACT_PORT, height=600, width="100%")

In [ ]:
# ── Alternative: Next.js inside Colab ────────────────────────────────────────
import subprocess, threading, time
from google.colab import output as colab_output

NEXT_PORT = 4001
NEXT_DIR  = "/content/my-next-app"

if not os.path.exists(NEXT_DIR):
    !npx create-next-app@latest my-next-app --ts --eslint --no-git --yes 2>/dev/null

def _next():
    subprocess.run(
        ["npm", "run", "dev", "--", "-p", str(NEXT_PORT)],
        cwd=NEXT_DIR, check=False,
    )

threading.Thread(target=_next, daemon=True).start()
time.sleep(8)

colab_output.serve_kernel_port_as_iframe(NEXT_PORT, height=600, width="100%")

---
## 📊 Cell 8 — Platform Health & Metrics
_Check the status of all running services._

In [ ]:
import requests, json

health = requests.get(f"{api_url}/v1/health").json()
models = requests.get(f"{api_url}/v1/models",
    headers={"X-API-Key": api_key}).json()

print("\n📊 Platform Health")
print("=" * 40)
for k, v in health.items():
    print(f"  {k:<22} : {v}")

print("\n🤖 Available Models")
print("=" * 40)
for m in models["data"]:
    print(f"  - {m['id']}")

---
## ♾️ Cell 9 — Keep Alive
_Prevents Colab from disconnecting due to inactivity._

In [ ]:
import time

print("Platform running — keeping session alive. Press STOP to terminate.")
print(f"API  : {api_url}")
print(f"UI   : {ui_url}")
print(f"Docs : {api_url}/docs")
print()

for i in range(10_000):
    time.sleep(300)
    print(f"= [{i+1} × 5min alive]", end="", flush=True)